# Get play counts from Spotify's web app

Update monthly
Add play counts to csv

In [2]:
import os
import json
from dotenv import load_dotenv
from datetime import date
import requests
import pandas as pd

load_dotenv()

SPOTIFY_CLIENT_ID = os.getenv('SPOTIFY_CLIENT_ID')
SPOTIFY_CLIENT_SECRET = os.getenv('SPOTIFY_CLIENT_SECRET')
SPOTIFY_HASH = os.getenv('SPOTIFY_HASH')

In [3]:
with open('consts.json', 'r') as file:
    data = json.load(file)

album = data['album']
artist = data['artist']

In [135]:
# Authenticate to Spotify and request an access token

url = 'https://accounts.spotify.com/api/token'

headers = {
    "Content-Type": "application/x-www-form-urlencoded"
}

payload = {
    "grant_type": "client_credentials",
    "client_id": SPOTIFY_CLIENT_ID,
    "client_secret": SPOTIFY_CLIENT_SECRET
}

res = requests.post(url, headers=headers, data=payload)
spotify_access_token = res.json()["access_token"]

In [136]:
# Get Client Token

headers = {
    'accept': 'application/json',
    'accept-language': 'en-US,en;q=0.7',
    'content-type': 'application/json',
    'origin': 'https://open.spotify.com',
    'priority': 'u=1, i',
    'referer': 'https://open.spotify.com/',
    'sec-ch-ua': '"Not=A?Brand";v="99", "Google Chrome";v="151", "Chromium";v="151"',
    'sec-ch-ua-mobile': '?0',
    'sec-ch-ua-platform': '"macOS"',
    'sec-fetch-dest': 'empty',
    'sec-fetch-mode': 'cors',
    'sec-fetch-site': 'same-site',
    'sec-gpc': '1',
    'user-agent': 'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/151.0.0.0 Safari/537.36',
}

json_data = {
    'client_data': {
        'client_version': '1.2.98.59.g81a1284c-development',
        'client_id': 'd8a5ed958d274c2e8ee717e6a4b0971d',
        'js_sdk_data': {
            'device_brand': 'Apple',
            'device_model': 'unknown',
            'os': 'macos',
            'os_version': '10.15.7',
            'device_id': '69bc79a0-043e-4d4c-a57d-d36b182f7517',
            'device_type': 'computer',
        },
    },
}

response = requests.post('https://clienttoken.spotify.com/v1/clienttoken', headers=headers, json=json_data)

In [137]:
if response.status_code == 200:
    client_token = response.json()['granted_token']['token']
    print(client_token)
else:
    print("Error", response)

AAFJ7zdlZc/qACOMKfpoJzPDjEIk9LRaPiJFjA9teEpeTl6+XkhERUKKlFJsT5gIzj72zZe/ctiWgElp43Nhg7ksF+LERq2AqAzinqAt0jcSGmsbA/AWuAcIiNd9v5mOqwK2zqK8C9/lC2d4WY/+6Es6A+0/ms2Srd4ZzuA7z/LFAUn0GpZ5HvOu3OZQDluSpnBZiRzmgMUXjcBuwEYcW/9xm5QHYc/WSwSMhuU3p8VZCwrwM6oC2PT/Tr0sfkJ97LbC0oz9VIfuhg34i1BCcwQACAA/+4s8taieTnBs5F08DUJERvmg3lnpAb6mFvTPwNWOAdppvr3xsTyFM4kp54KlTnE7UBLmA211aMKdFz654/0Zs5Xuyn8=


In [138]:
# New required headers for posting to Spotify's undocumented api
# https://api-partner.spotify.com/pathfinder/v2/query

headers = {
    'accept': 'application/json',
    'accept-language': 'en',
    'app-platform': 'WebPlayer',
    'authorization': f'Bearer {spotify_access_token}',
    'client-token': client_token,
    'content-type': 'application/json;charset=UTF-8',
    'origin': 'https://open.spotify.com',
    'priority': 'u=1, i',
    'referer': 'https://open.spotify.com/',
    'sec-ch-ua': '"Not=A?Brand";v="99", "Google Chrome";v="151", "Chromium";v="151"',
    'sec-ch-ua-mobile': '?0',
    'sec-ch-ua-platform': '"macOS"',
    'sec-fetch-dest': 'empty',
    'sec-fetch-mode': 'cors',
    'sec-fetch-site': 'same-site',
    'sec-gpc': '1',
    'spotify-app-version': '1.2.98.59.g81a1284c-development',
    'user-agent': 'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/151.0.0.0 Safari/537.36',
}

In [113]:
# Read in Spotify track id's from the audio-features csv
script_dir = os.path.dirname(os.path.abspath('get-play-counts.ipynb'))
audio_feats_file_path = os.path.join(script_dir, '..', 'albums', f'{album}', 'output', f'audio_feats_{album}.csv')
df = pd.read_csv(audio_feats_file_path)

In [142]:
# Loop through Spotify track ids and hit the undocumented endpoint
play_counts = []
play_count_dates = []

for track_id in df["spotify_track_id"]:
    url = 'https://api-partner.spotify.com/pathfinder/v2/query'
    json_data = {
        'variables': {
            'uri': f'spotify:track:{track_id}',
            'includeVideoAssociationItems': False,
        },
        'operationName': 'getTrack',
        'extensions': {
            'persistedQuery': {
                'version': 1,
                'sha256Hash': SPOTIFY_HASH,
            },
        },
    }

    response = requests.post(url, headers=headers, json=json_data)
    if response.status_code == 200:
        play_count = response.json()["data"]["trackUnion"]["playcount"]

        play_counts.append(play_count)
        play_count_dates.append(str(date.today()))
    else:
        print("Error", response.text)

Error {
  "error": {
    "status": 401,
    "message": "Missing/invalid/expired access token"
  }
}

Error {
  "error": {
    "status": 401,
    "message": "Missing/invalid/expired access token"
  }
}

Error {
  "error": {
    "status": 401,
    "message": "Missing/invalid/expired access token"
  }
}

Error {
  "error": {
    "status": 401,
    "message": "Missing/invalid/expired access token"
  }
}

Error {
  "error": {
    "status": 401,
    "message": "Missing/invalid/expired access token"
  }
}

Error {
  "error": {
    "status": 401,
    "message": "Missing/invalid/expired access token"
  }
}

Error {
  "error": {
    "status": 401,
    "message": "Missing/invalid/expired access token"
  }
}

Error {
  "error": {
    "status": 401,
    "message": "Missing/invalid/expired access token"
  }
}

Error {
  "error": {
    "status": 401,
    "message": "Missing/invalid/expired access token"
  }
}

Error {
  "error": {
    "status": 401,
    "message": "Missing/invalid/expired access toke

In [124]:
play_counts

[]

In [103]:
df[f"play_count"] = play_counts
df["play_count_last_updated"] = play_count_dates

In [104]:
df

,spotify_track_id,track_name,track_number,id,href,isrc,acousticness,danceability,energy,instrumentalness,key,liveness,loudness,mode,speechiness,tempo,valence,play_count,play_count_last_updated
0,2AaB2ZDeJXu6j4Csos4gZH,Time Travelin' (A Tribute To Fela),1,6382aa7b-2fdb-4251-8dc5-18903a522332,https://open.spotify.com/track/2AaB2ZDeJXu6j4C...,USMC10000119,0.2270,0.830,0.551,0.254000,7,0.3760,-14.559,1,0.283,104.179,0.434,1541599,2026-08-16
1,1ZD3CMagZCxFyrc2zPyvBl,Heat,2,d3f7c3de-043e-4242-b630-7ec5af6083f0,https://open.spotify.com/track/1ZD3CMagZCxFyrc...,USMC10000120,0.0057,0.811,0.732,0.031200,11,0.0929,-7.814,0,0.262,102.587,0.627,1467353,2026-08-16
2,6Is1oWZB3Ry1bbMx2MKeui,Cold Blooded,3,cbd193e9-63ec-4fca-b460-48bf0973e87f,https://open.spotify.com/track/6Is1oWZB3Ry1bbM...,USMC10000121,0.3400,0.717,0.903,0.000180,4,0.1090,-6.395,0,0.351,98.829,0.456,1323990,2026-08-16
3,7bEEzWWgJS4HhzYtNLCXfa,Dooinit,4,aac586dc-d7de-408d-8580-d00a5cc52af6,https://open.spotify.com/track/7bEEzWWgJS4HhzY...,USMC10000122,0.1020,0.850,0.637,0.000000,5,0.0995,-5.820,0,0.393,92.859,0.760,2289770,2026-08-16
4,5NiUrZVKyLpsyj62Roq5FW,The Light,5,21901fdd-9963-4813-89d4-29238a40df59,https://open.spotify.com/track/5NiUrZVKyLpsyj6...,USMC10000123,0.0343,0.939,0.727,0.000000,4,0.0855,-4.349,0,0.235,96.968,0.660,31665641,2026-08-16
5,3aoSjb0bYD2pR2h7R4UzAj,Funky For You,6,f6072dab-961b-468d-86ed-9b3b3bae6b31,https://open.spotify.com/track/3aoSjb0bYD2pR2h...,USMC10000124,0.1380,0.542,0.706,0.000020,11,0.1660,-6.678,1,0.542,98.674,0.689,4959181,2026-08-16
6,12DQLP0EURlKcwguEJM5oY,The Questions,7,07a320f6-a73c-4ab6-97b7-095828d94163,https://open.spotify.com/track/12DQLP0EURlKcwg...,USMC10000125,0.3700,0.828,0.388,0.000000,7,0.3960,-12.355,1,0.333,90.742,0.820,3180740,2026-08-16
7,3cwWEfm58FQbdJBM4BLI43,Time Travelin Reprise,8,87fd629f-4015-4c9d-ab2c-afd4554c2426,https://open.spotify.com/track/3cwWEfm58FQbdJB...,USMC10000126,0.0168,0.904,0.332,0.819000,7,0.3200,-15.613,1,0.194,104.232,0.690,816925,2026-08-16
8,6OE9S6XF0U1lNfeaUNSjYl,The 6th Sense,9,421b4d21-7b24-4ca9-b695-13e8dca916af,https://open.spotify.com/track/6OE9S6XF0U1lNfe...,USMC10000134,0.0746,0.687,0.827,0.000000,1,0.7020,-6.301,1,0.377,94.441,0.882,8484551,2026-08-16
9,71gd3cOuLvtmUb17OARvsJ,A Film Called (Pimp),10,f277a79e-6496-4433-9f02-2c74d11e68db,https://open.spotify.com/track/71gd3cOuLvtmUb1...,USMC10000127,0.0915,0.788,0.664,0.000005,10,0.5080,-5.047,0,0.372,78.294,0.763,1061552,2026-08-16


In [ ]:
output_file_path = os.path.join(script_dir, '..', 'albums', f'{album}', 'output', f'audio_feats_and_play_counts_{album}.csv')

In [106]:
df.to_csv(output_file_path, index=False)